In [127]:
# import libraries
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None) # display all columns in the dataframe

In [128]:
patients = pd.read_csv('../data/processed/patients_cleaned.csv', parse_dates=['registration_date'])
vitals = pd.read_csv('../data/processed/vital_signs_cleaned.csv', parse_dates=['timestamp'])
history = pd.read_csv('../data/processed/clinical_history_cleaned.csv')
labs = pd.read_csv('../data/processed/laboratory_results_cleaned.csv', parse_dates=['timestamp'])
outcomes = pd.read_csv('../data/processed/sepsis_outcomes_cleaned.csv', parse_dates=['diagnosis_time'])

tables = {'patients': patients, 'vitals': vitals, 'history': history, 'labs': labs, 'outcomes': outcomes}
for name, df in tables.items():
    print(f"{name} shape: {df.shape}")

patients shape: (600, 5)
vitals shape: (11807, 8)
history shape: (1449, 6)
labs shape: (2430, 8)
outcomes shape: (600, 6)


In [129]:
# Sort vitals and labs dataframes by patient_id and timestamp
vitals = vitals.sort_values(['patient_id', 'timestamp']).reset_index(drop=True)
labs = labs.sort_values(['patient_id', 'timestamp']).reset_index(drop=True)

In [130]:
vitals.columns

Index(['observation_id', 'patient_id', 'timestamp', 'heart_rate',
       'temperature', 'oxygen_saturation', 'respiratory_rate',
       'blood_pressure'],
      dtype='str')

In [131]:
labs.columns

Index(['lab_id', 'patient_id', 'timestamp', 'white_cell_count', 'crp',
       'lactate', 'creatinine', 'platelet_count'],
      dtype='str')

In [132]:
vital_cols = [
  'heart_rate',
  'temperature',
  'oxygen_saturation',
  'respiratory_rate',
  'blood_pressure'
]

lab_cols = [
  'white_cell_count',
  'crp',
  'lactate',
  'creatinine',
  'platelet_count',
]

In [133]:
outcomes.columns

Index(['outcome_id', 'patient_id', 'sepsis_event', 'diagnosis_time',
       'hospitalisation_required', 'outcome_status'],
      dtype='str')

In [134]:
# Feature engineering for prediction time
rng = np.random.default_rng(7) # set random seed for reproducibility

def get_prediction_time(row, vitals_df):
  if row['sepsis_event']:
    return row['diagnosis_time'] - pd.Timedelta(hours=9)
  pv = vitals_df[vitals_df['patient_id'] == row['patient_id']]
  start, end = pv['timestamp'].min(), pv['timestamp'].max()
  span_hours = max((end - start).total_seconds() / 3600, 1)
  offset_hours = rng.uniform(0.4, 0.9) * span_hours
  return start + pd.Timedelta(hours=offset_hours)

outcomes = outcomes.copy()
outcomes['prediction_time'] = outcomes.apply(get_prediction_time, vitals_df=vitals, axis=1)
outcomes[['patient_id', 'sepsis_event', 'diagnosis_time', 'prediction_time']].head(15)

,patient_id,sepsis_event,diagnosis_time,prediction_time
0,1,False,NaT,2024-10-21 22:31:02.552633471
1,2,False,NaT,2025-06-26 10:29:09.302720280
2,3,False,NaT,2024-03-01 02:25:35.501601845
3,4,False,NaT,2024-08-02 16:10:19.745594826
4,5,False,NaT,2024-07-10 00:00:48.032376176
5,6,False,NaT,2024-10-17 12:01:52.909136091
6,7,False,NaT,2025-12-17 07:05:33.790790426
7,8,False,NaT,2024-08-30 14:15:42.264998141
8,9,True,2024-06-01 22:58:00,2024-06-01 13:58:00.000000000
9,10,False,NaT,2024-03-31 10:06:58.001875856


In [135]:
# Feature engineering for vitals

LOOKBACK_HOURS = 6

def vitals_features(pid, cutoff, df):
    window = df[
        (df['patient_id'] == pid)
        & (df['timestamp'] <= cutoff)
        & (df['timestamp'] >= cutoff - pd.Timedelta(hours=LOOKBACK_HOURS))
    ].sort_values('timestamp')

    feats = {
        'vitals_observation_count_6h': len(window),
        'hours_since_last_vital': (
            (cutoff - window['timestamp'].iloc[-1]).total_seconds() / 3600
            if not window.empty else np.nan
        )
    }

    for col in vital_cols:
        observed = window.dropna(subset=[col])
        vals = observed[col]

        feats[f'{col}_mean_6h'] = vals.mean()
        feats[f'{col}_min_6h'] = vals.min()
        feats[f'{col}_max_6h'] = vals.max()
        feats[f'{col}_last_6h'] = vals.iloc[-1] if len(vals) else np.nan

        if len(vals) > 1:
            feats[f'{col}_std_6h'] = vals.std()

            hours = (
                observed['timestamp'].iloc[-1]
                - observed['timestamp'].iloc[0]
            ).total_seconds() / 3600

            feats[f'{col}_rate_per_hour_6h'] = (
                (vals.iloc[-1] - vals.iloc[0]) / hours
                if hours > 0 else 0.0
            )
        elif len(vals) == 1:
            feats[f'{col}_std_6h'] = 0.0
            feats[f'{col}_rate_per_hour_6h'] = 0.0
        else:
            feats[f'{col}_std_6h'] = np.nan
            feats[f'{col}_rate_per_hour_6h'] = np.nan

    # missingness flags for each vital sign in the window
    for col in vital_cols:
        feats[f'{col}_missing_6h'] = int(window[col].isna().all())

    return feats


vitals_features_rows = [
    {'patient_id': pid, **vitals_features(pid, prediction_time, vitals)}
    for pid, prediction_time in zip(
        outcomes['patient_id'],
        outcomes['prediction_time']
    )
]

vitals_features_df = pd.DataFrame(vitals_features_rows)
vitals_features_df.head(10)


,patient_id,vitals_observation_count_6h,hours_since_last_vital,heart_rate_mean_6h,heart_rate_min_6h,heart_rate_max_6h,heart_rate_last_6h,heart_rate_std_6h,heart_rate_rate_per_hour_6h,temperature_mean_6h,temperature_min_6h,temperature_max_6h,temperature_last_6h,temperature_std_6h,temperature_rate_per_hour_6h,oxygen_saturation_mean_6h,oxygen_saturation_min_6h,oxygen_saturation_max_6h,oxygen_saturation_last_6h,oxygen_saturation_std_6h,oxygen_saturation_rate_per_hour_6h,respiratory_rate_mean_6h,respiratory_rate_min_6h,respiratory_rate_max_6h,respiratory_rate_last_6h,respiratory_rate_std_6h,respiratory_rate_rate_per_hour_6h,blood_pressure_mean_6h,blood_pressure_min_6h,blood_pressure_max_6h,blood_pressure_last_6h,blood_pressure_std_6h,blood_pressure_rate_per_hour_6h,heart_rate_missing_6h,temperature_missing_6h,oxygen_saturation_missing_6h,respiratory_rate_missing_6h,blood_pressure_missing_6h
0,1,1,1.900709,84.200000,84.2,84.2,84.2,0.000000,0.000000,36.800000,36.80,36.80,36.80,0.000000,0.000000,98.900000,98.9,98.9,98.9,0.000000,0.000000,14.300000,14.3,14.3,14.3,0.000000,0.000000,114.800000,114.8,114.8,114.8,0.000000,0.000000,0,0,0,0,0
1,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,1,1
2,3,2,3.143195,75.300000,72.5,78.1,78.1,3.959798,6.000000,36.435000,36.37,36.50,36.37,0.091924,-0.139286,95.300000,95.1,95.5,95.5,0.282843,0.428571,15.250000,15.0,15.5,15.5,0.353553,0.535714,130.250000,125.6,134.9,125.6,6.576093,-9.964286,0,0,0,0,0
3,4,1,1.455485,72.500000,72.5,72.5,72.5,0.000000,0.000000,36.490000,36.49,36.49,36.49,0.000000,0.000000,96.900000,96.9,96.9,96.9,0.000000,0.000000,16.400000,16.4,16.4,16.4,0.000000,0.000000,119.000000,119.0,119.0,119.0,0.000000,0.000000,0,0,0,0,0
4,5,6,0.030009,74.233333,69.9,79.3,69.9,3.149391,-1.616046,36.793333,36.71,36.94,36.71,0.096885,-0.029226,97.316667,96.8,97.9,97.2,0.416733,-0.017192,12.783333,11.7,13.9,12.9,0.760044,-0.034384,138.250000,128.8,148.6,128.8,7.109923,-3.404011,0,0,0,0,0
5,6,1,4.914697,78.900000,78.9,78.9,78.9,0.000000,0.000000,37.010000,37.01,37.01,37.01,0.000000,0.000000,97.300000,97.3,97.3,97.3,0.000000,0.000000,22.000000,22.0,22.0,22.0,0.000000,0.000000,123.200000,123.2,123.2,123.2,0.000000,0.000000,0,0,0,0,0
6,7,1,4.292720,64.300000,64.3,64.3,64.3,0.000000,0.000000,37.520000,37.52,37.52,37.52,0.000000,0.000000,96.200000,96.2,96.2,96.2,0.000000,0.000000,15.000000,15.0,15.0,15.0,0.000000,0.000000,121.100000,121.1,121.1,121.1,0.000000,0.000000,0,0,0,0,0
7,8,3,0.395074,80.233333,76.9,83.7,80.1,3.401960,0.897196,36.610000,36.23,37.04,36.56,0.407308,0.092523,97.366667,96.2,98.6,97.3,1.201388,-0.364486,14.700000,14.2,15.3,14.2,0.556776,-0.308411,124.733333,120.8,128.1,128.1,3.682843,2.046729,0,0,0,0,0
8,9,2,0.700000,73.000000,69.6,76.4,76.4,4.808326,4.800000,37.215000,37.09,37.34,37.34,0.176777,0.176471,97.150000,96.2,98.1,96.2,1.343503,-1.341176,15.700000,14.2,17.2,17.2,2.121320,2.117647,128.150000,120.1,136.2,120.1,11.384419,-11.364706,0,0,0,0,0
9,10,2,2.616112,84.950000,82.5,87.4,82.5,3.464823,-1.709302,36.195000,35.85,36.54,35.85,0.487904,-0.240698,98.150000,98.0,98.3,98.3,0.212132,0.104651,15.850000,14.3,17.4,17.4,2.192031,1.081395,116.450000,114.2,118.7,118.7,3.181981,1.569767,0,0,0,0,0


In [136]:
# Generate labs features for each patient at their prediction time
LAB_LOOKBACK_HOURS = 24

def labs_features(pid, cutoff, df):
  window = df[
    (df['patient_id'] == pid) & 
    (df['timestamp'] <= cutoff) & 
    (df['timestamp'] >= cutoff - pd.Timedelta(hours=LAB_LOOKBACK_HOURS))
  ].sort_values('timestamp')

  feats = {
    'labs_observation_count_24h': len(window),
    'hours_since_last_lab': (
      (cutoff - window['timestamp'].iloc[-1]).total_seconds() / 3600
      if not window.empty else np.nan
    )
  }

  for col in lab_cols:
    observed = window.dropna(subset=[col])
    vals = observed[col]

    feats[f'{col}_mean_24h'] = vals.mean()
    feats[f'{col}_min_24h'] = vals.min()
    feats[f'{col}_max_24h'] = vals.max()
    feats[f'{col}_last_24h'] = vals.iloc[-1] if len(vals) else np.nan

    if len(vals) > 1:
      feats[f'{col}_std_24h'] = vals.std()

      hours = (
        observed['timestamp'].iloc[-1]
        - observed['timestamp'].iloc[0]
      ).total_seconds() / 3600

      feats[f'{col}_rate_per_hour_24h'] = (
        (vals.iloc[-1] - vals.iloc[0]) / hours
        if hours > 0 else 0.0
      )
    elif len(vals) == 1:
      feats[f'{col}_std_24h'] = 0.0
      feats[f'{col}_rate_per_hour_24h'] = 0.0
    else:
      feats[f'{col}_std_24h'] = np.nan
      feats[f'{col}_rate_per_hour_24h'] = np.nan

  # missingness flags for each lab in the window
  for col in lab_cols:
    feats[f'{col}_missing_24h'] = int(window[col].isna().all())
    
  return feats 
  
labs_features_rows = [
  {'patient_id': pid, **labs_features(pid, cutoff, labs)}
  for pid, cutoff in zip(outcomes['patient_id'], outcomes['prediction_time'])
]
labs_features_df = pd.DataFrame(labs_features_rows)
labs_features_df.head(10)

,patient_id,labs_observation_count_24h,hours_since_last_lab,white_cell_count_mean_24h,white_cell_count_min_24h,white_cell_count_max_24h,white_cell_count_last_24h,white_cell_count_std_24h,white_cell_count_rate_per_hour_24h,crp_mean_24h,crp_min_24h,crp_max_24h,crp_last_24h,crp_std_24h,crp_rate_per_hour_24h,lactate_mean_24h,lactate_min_24h,lactate_max_24h,lactate_last_24h,lactate_std_24h,lactate_rate_per_hour_24h,creatinine_mean_24h,creatinine_min_24h,creatinine_max_24h,creatinine_last_24h,creatinine_std_24h,creatinine_rate_per_hour_24h,platelet_count_mean_24h,platelet_count_min_24h,platelet_count_max_24h,platelet_count_last_24h,platelet_count_std_24h,platelet_count_rate_per_hour_24h,white_cell_count_missing_24h,crp_missing_24h,lactate_missing_24h,creatinine_missing_24h,platelet_count_missing_24h
0,1,2,11.917376,6.700,6.70,6.70,6.70,0.000000,0.000000,10.25,8.0,12.5,8.0,3.181981,-0.747922,0.840,0.83,0.85,0.85,0.014142,0.003324,0.815,0.78,0.85,0.78,0.049497,-0.011634,303.0,301.0,305.0,301.0,2.828427,-0.664820,0,0,0,0,0
1,2,2,7.369251,7.715,7.61,7.82,7.61,0.148492,-0.036311,15.50,14.1,16.9,16.9,1.979899,0.484150,0.605,0.54,0.67,0.54,0.091924,-0.022478,0.870,0.84,0.90,0.90,0.042426,0.010375,187.5,180.0,195.0,180.0,10.606602,-2.593660,0,0,0,0,0
2,3,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,1,1
3,4,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,1,1
4,5,5,6.396676,4.792,4.12,5.26,4.12,0.477776,-0.103794,6.66,3.5,8.4,6.7,2.010721,-0.154780,0.978,0.87,1.15,0.97,0.105688,0.009105,0.514,0.37,0.75,0.75,0.164560,0.034598,247.0,232.0,258.0,258.0,10.723805,1.638847,0,0,0,0,0
5,6,1,18.614697,7.630,7.63,7.63,7.63,0.000000,0.000000,0.50,0.5,0.5,0.5,0.000000,0.000000,0.620,0.62,0.62,0.62,0.000000,0.000000,1.340,1.34,1.34,1.34,0.000000,0.000000,332.0,332.0,332.0,332.0,0.000000,0.000000,0,0,0,0,0
6,7,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,1,1
7,8,1,5.595074,9.840,9.84,9.84,9.84,0.000000,0.000000,4.90,4.9,4.9,4.9,0.000000,0.000000,0.690,0.69,0.69,0.69,0.000000,0.000000,0.860,0.86,0.86,0.86,0.000000,0.000000,243.0,243.0,243.0,243.0,0.000000,0.000000,0,0,0,0,0
8,9,1,6.550000,9.860,9.86,9.86,9.86,0.000000,0.000000,0.50,0.5,0.5,0.5,0.000000,0.000000,1.240,1.24,1.24,1.24,0.000000,0.000000,0.770,0.77,0.77,0.77,0.000000,0.000000,239.0,239.0,239.0,239.0,0.000000,0.000000,0,0,0,0,0
9,10,1,3.032778,6.220,6.22,6.22,6.22,0.000000,0.000000,4.20,4.2,4.2,4.2,0.000000,0.000000,1.100,1.10,1.10,1.10,0.000000,0.000000,1.030,1.03,1.03,1.03,0.000000,0.000000,298.0,298.0,298.0,298.0,0.000000,0.000000,0,0,0,0,0


In [137]:
patients.columns

Index(['patient_id', 'age', 'gender', 'medical_conditions',
       'registration_date'],
      dtype='str')

In [138]:
# Create a static features dataframe with patient_id, age, gender
static = patients[['patient_id', 'age', 'gender']].copy()

def count_comorbidities(value):
  if pd.isna(value) or value == 'None reported':
    return 0
  return len([item.strip() for item in str(value).split(',') if item.strip()])

static['comorbidity_count'] = patients['medical_conditions'].apply(count_comorbidities)

def count_history_items(value):
  if pd.isna(value) or str(value).strip().lower() in {
    '',
    'none reported',
    'unknown'
  }:
    return 0

  return len([
    item.strip() for item in str(value).split(',') if item.strip()
  ])

history_features = (
  history.groupby('patient_id', as_index=False).agg(
    history_record_count=('patient_id', 'size'),
    diagnosis_history_count=(
      'diagnosis_history', 
      lambda values: sum(count_history_items(value) for value in values)
    ),
    infection_history_count=(
      'infection_history', 
      lambda values: sum(count_history_items(value) for value in values)
    ),
    medication_history_count=(
      'medication_history', 
      lambda values: sum(count_history_items(value) for value in values)
    ),
    treatment_history_count=(
      'treatment_history', 
      lambda values: sum(count_history_items(value) for value in values)
    )
  )
)

static = static.merge(history_features, on='patient_id', how='left')
  
static = pd.get_dummies(static, columns=['gender'], drop_first=True)
static['age_65_or_over'] = (static['age'] >= 65).astype(int)
static.head(10)

,patient_id,age,comorbidity_count,history_record_count,diagnosis_history_count,infection_history_count,medication_history_count,treatment_history_count,gender_Male,gender_Other/Not specified,age_65_or_over
0,1,66,2,1,1,1,1,1,True,False,1
1,2,42,3,2,2,2,2,2,False,False,0
2,3,74,1,2,2,2,2,1,True,False,1
3,4,77,2,1,1,1,1,1,False,False,1
4,5,25,0,2,2,2,2,2,False,False,0
5,6,37,0,4,3,4,3,3,False,False,0
6,7,63,1,3,3,3,3,2,False,False,0
7,8,55,0,3,3,3,3,3,True,False,0
8,9,60,2,4,4,4,4,3,True,False,0
9,10,45,1,1,1,1,1,1,True,False,0


In [139]:
# Combine static, vitals, labs, and outcomes into a single feature dataframe
feature = (
  static
  .merge(vitals_features_df, on='patient_id')
  .merge(labs_features_df, on='patient_id')
  .merge(outcomes[['patient_id', 'sepsis_event']], on='patient_id')
)


In [140]:
# clinical threshold flags
def threshold_flag(series, condition):
  return pd.Series(
    np.where(series.notna(), condition(series).astype(int), np.nan),
    index=series.index 
  )

feature['heart_rate_high_6h'] = threshold_flag(
  feature['heart_rate_max_6h'], 
  lambda values: values >= 100
)

feature['fever_6h'] = threshold_flag(
  feature['temperature_last_6h'],
  lambda values: values >= 38.0
)

feature['low_oxygen_saturation_6h'] = threshold_flag(
  feature['oxygen_saturation_last_6h'],
  lambda values: values < 94.0
)

feature['high_respiratory_rate_6h'] = threshold_flag(
  feature['respiratory_rate_max_6h'],
  lambda values: values >= 22
)

feature['low_blood_pressure_6h'] = threshold_flag(
  feature['blood_pressure_last_6h'],
  lambda values: values < 90
)

In [141]:

feature_columns = [
  column for column in feature.columns
  if column not in (
      'patient_id', 
      'sepsis_event'
    )
]

numeric_cols = feature[feature_columns].select_dtypes(
    include='number'
  ).columns

for column in numeric_cols:
    if feature[column].isna().any():
        feature[f'{column}_missing'] = feature[column].isna().astype(int)

feature_columns = [
  column for column in feature.columns
  if column not in (
      'patient_id', 
      'sepsis_event'
    )
]

print(feature.shape)

(600, 158)


In [142]:
feature.head(10)

,patient_id,age,comorbidity_count,history_record_count,diagnosis_history_count,infection_history_count,medication_history_count,treatment_history_count,gender_Male,gender_Other/Not specified,age_65_or_over,vitals_observation_count_6h,hours_since_last_vital,heart_rate_mean_6h,heart_rate_min_6h,heart_rate_max_6h,heart_rate_last_6h,heart_rate_std_6h,heart_rate_rate_per_hour_6h,temperature_mean_6h,temperature_min_6h,temperature_max_6h,temperature_last_6h,temperature_std_6h,temperature_rate_per_hour_6h,oxygen_saturation_mean_6h,oxygen_saturation_min_6h,oxygen_saturation_max_6h,oxygen_saturation_last_6h,oxygen_saturation_std_6h,oxygen_saturation_rate_per_hour_6h,respiratory_rate_mean_6h,respiratory_rate_min_6h,respiratory_rate_max_6h,respiratory_rate_last_6h,respiratory_rate_std_6h,respiratory_rate_rate_per_hour_6h,blood_pressure_mean_6h,blood_pressure_min_6h,blood_pressure_max_6h,blood_pressure_last_6h,blood_pressure_std_6h,blood_pressure_rate_per_hour_6h,heart_rate_missing_6h,temperature_missing_6h,oxygen_saturation_missing_6h,respiratory_rate_missing_6h,blood_pressure_missing_6h,labs_observation_count_24h,hours_since_last_lab,white_cell_count_mean_24h,white_cell_count_min_24h,white_cell_count_max_24h,white_cell_count_last_24h,white_cell_count_std_24h,white_cell_count_rate_per_hour_24h,crp_mean_24h,crp_min_24h,crp_max_24h,crp_last_24h,crp_std_24h,crp_rate_per_hour_24h,lactate_mean_24h,lactate_min_24h,lactate_max_24h,lactate_last_24h,lactate_std_24h,lactate_rate_per_hour_24h,creatinine_mean_24h,creatinine_min_24h,creatinine_max_24h,creatinine_last_24h,creatinine_std_24h,creatinine_rate_per_hour_24h,platelet_count_mean_24h,platelet_count_min_24h,platelet_count_max_24h,platelet_count_last_24h,platelet_count_std_24h,platelet_count_rate_per_hour_24h,white_cell_count_missing_24h,crp_missing_24h,lactate_missing_24h,creatinine_missing_24h,platelet_count_missing_24h,sepsis_event,heart_rate_high_6h,fever_6h,low_oxygen_saturation_6h,high_respiratory_rate_6h,low_blood_pressure_6h,hours_since_last_vital_missing,heart_rate_mean_6h_missing,heart_rate_min_6h_missing,heart_rate_max_6h_missing,heart_rate_last_6h_missing,heart_rate_std_6h_missing,heart_rate_rate_per_hour_6h_missing,temperature_mean_6h_missing,temperature_min_6h_missing,temperature_max_6h_missing,temperature_last_6h_missing,temperature_std_6h_missing,temperature_rate_per_hour_6h_missing,oxygen_saturation_mean_6h_missing,oxygen_saturation_min_6h_missing,oxygen_saturation_max_6h_missing,oxygen_saturation_last_6h_missing,oxygen_saturation_std_6h_missing,oxygen_saturation_rate_per_hour_6h_missing,respiratory_rate_mean_6h_missing,respiratory_rate_min_6h_missing,respiratory_rate_max_6h_missing,respiratory_rate_last_6h_missing,respiratory_rate_std_6h_missing,respiratory_rate_rate_per_hour_6h_missing,blood_pressure_mean_6h_missing,blood_pressure_min_6h_missing,blood_pressure_max_6h_missing,blood_pressure_last_6h_missing,blood_pressure_std_6h_missing,blood_pressure_rate_per_hour_6h_missing,hours_since_last_lab_missing,white_cell_count_mean_24h_missing,white_cell_count_min_24h_missing,white_cell_count_max_24h_missing,white_cell_count_last_24h_missing,white_cell_count_std_24h_missing,white_cell_count_rate_per_hour_24h_missing,crp_mean_24h_missing,crp_min_24h_missing,crp_max_24h_missing,crp_last_24h_missing,crp_std_24h_missing,crp_rate_per_hour_24h_missing,lactate_mean_24h_missing,lactate_min_24h_missing,lactate_max_24h_missing,lactate_last_24h_missing,lactate_std_24h_missing,lactate_rate_per_hour_24h_missing,creatinine_mean_24h_missing,creatinine_min_24h_missing,creatinine_max_24h_missing,creatinine_last_24h_missing,creatinine_std_24h_missing,creatinine_rate_per_hour_24h_missing,platelet_count_mean_24h_missing,platelet_count_min_24h_missing,platelet_count_max_24h_missing,platelet_count_last_24h_missing,platelet_count_std_24h_missing,platelet_count_rate_per_hour_24h_missing,heart_rate_high_6h_missing,fever_6h_missing,low_oxygen_saturation_6h_missing,high_respiratory_rate_6h_missing,low_blood_pre

In [143]:
feature.columns

Index(['patient_id', 'age', 'comorbidity_count', 'history_record_count',
       'diagnosis_history_count', 'infection_history_count',
       'medication_history_count', 'treatment_history_count', 'gender_Male',
       'gender_Other/Not specified',
       ...
       'platelet_count_min_24h_missing', 'platelet_count_max_24h_missing',
       'platelet_count_last_24h_missing', 'platelet_count_std_24h_missing',
       'platelet_count_rate_per_hour_24h_missing',
       'heart_rate_high_6h_missing', 'fever_6h_missing',
       'low_oxygen_saturation_6h_missing', 'high_respiratory_rate_6h_missing',
       'low_blood_pressure_6h_missing'],
      dtype='str', length=158)

In [144]:
# Check mean values of selected features grouped by sepsis event
check_cols = ['heart_rate_last_6h', 'oxygen_saturation_last_6h', 'crp_last_24h', 'lactate_last_24h']
feature.groupby('sepsis_event')[check_cols].mean()

,heart_rate_last_6h,oxygen_saturation_last_6h,crp_last_24h,lactate_last_24h
sepsis_event,,,,
False,78.404859,97.483376,6.341192,1.005881
True,82.007273,97.167273,8.567273,1.051818


In [145]:
# Compute correlation of features with sepsis_event
corr = (
  feature[feature_columns + ['sepsis_event']]
  .corr()['sepsis_event']
  .drop('sepsis_event')
)

corr.sort_values(key=abs, ascending=False).head(10)

blood_pressure_last_6h     -0.192235
crp_max_24h                 0.182717
blood_pressure_min_6h      -0.173874
crp_std_24h                 0.169601
crp_last_24h                0.164869
oxygen_saturation_std_6h    0.163329
heart_rate_std_6h           0.161213
blood_pressure_std_6h       0.150608
platelet_count_mean_24h    -0.145511
platelet_count_min_24h     -0.145433
Name: sepsis_event, dtype: float64

In [146]:
# Save the final feature dataframe to a CSV file
feature.to_csv('../data/processed/sepsis_features_2.csv', index=False)